In [1]:
import argparse
import os
import pickle
import time

from importlib import metadata
import torch
try:
    try:
        if metadata.version("rsl-rl"):
            raise ImportError
    except metadata.PackageNotFoundError:
        if metadata.version("rsl-rl-lib") != "3.1.1":  #2.2.4
            raise ImportError
except (metadata.PackageNotFoundError, ImportError) as e:
    raise ImportError("Please uninstall 'rsl_rl' and install 'rsl-rl-lib==2.2.4'.") from e
from rsl_rl.runners import OnPolicyRunner

In [2]:
from bp000_env_cnoid import BP000Env as RLEnv

In [3]:
# 任意設定項目
# exp_name = 'collision-walking-rand'  # ckpt = 4000
# exp_name = 'friction-walking-fractal-kp2000kd50'
# exp_name = 'friction-walking-terrain2-kp2000kd50-relinvel20'  # ckpt = 20000
# exp_name = 'friction-walking-terrain1-kp4000kd50-linvel20-correct0.1-angvel4-plus-0.5'  # ckpt = 20000
# exp_name = 'friction-walking-terrain2-kp2000kd50-kpkdrand'
exp_name = 'friction-walking-terrain2-kp2000kd50-kpkdrand-5'
ckpt = 100

action_scale = 1.0 # 動作のスケールを調整

In [4]:
# 既存のセルを置き換え
import pandas as pd
import numpy as np

# データ収集用のリスト
# action_data = []
obs_data = []
torque_data = []
step_data = []

# CSVファイルの準備
csv_filename = f'obs_data/{exp_name}_step_data.csv'
os.makedirs('obs_data', exist_ok=True)

In [5]:
def _obs_vec(obs):
    # TensorDict or dict → 'policy' を優先
    if isinstance(obs, dict) or hasattr(obs, "get"):
        if "policy" in obs:
            obs = obs["policy"]
    if torch.is_tensor(obs):
        return obs.detach().cpu().numpy().ravel()
    return np.asarray(obs, dtype=np.float32).ravel()

In [6]:
## set robot path fix collisiton 
ROOT = os.path.abspath(os.path.join(os.getcwd(), ".."))  # /userdir
robot_path = os.path.join(ROOT, "userdir", "humanoid_research_k", "robots", "kawada_base.simple_collision.urdf")

In [7]:
log_dir = f"logs/{exp_name}"
env_cfg, obs_cfg, reward_cfg, command_cfg, train_cfg = pickle.load(open(f"logs/{exp_name}/cfgs.pkl", "rb"))
reward_cfg["reward_scales"] = {}

In [8]:
## override
# env_cfg["episode_length_s"] = 20.0
# command_cfg["lin_vel_x_range"] = [0.5, 0.5]
# env_cfg['dt'] = 0.001
# env_cfg['substeps'] = 10
# env_cfg["kd"] = 50
env_cfg['base_roll_noise'] = [0,0]
env_cfg['base_pitch_noise'] = [0,0]
env_cfg['termination_if_roll_greater_than'] = 150
env_cfg['termination_if_pitch_greater_than'] = 150
env_cfg['rotorInertia'] = 0.1
env_cfg["base_init_pos"] = [0.0, 0.0, 0.64]

In [9]:
reward_cfg

{'tracking_sigma': 0.25,
 'base_height_target': 0.64,
 'feet_height_target': 0.075,
 'reward_scales': {}}

In [10]:
env_cfg



{'num_actions': 12,
 'default_joint_angles': {'R_HIP_Y': 0.0,
  'R_HIP_R': 0.0,
  'R_HIP_P': -0.8,
  'R_KNEE': 1.6,
  'R_ANKLE_P': -0.8,
  'R_ANKLE_R': 0.0,
  'L_HIP_Y': 0.0,
  'L_HIP_R': 0.0,
  'L_HIP_P': -0.8,
  'L_KNEE': 1.6,
  'L_ANKLE_P': -0.8,
  'L_ANKLE_R': 0.0},
 'joint_names': ['R_HIP_Y',
  'R_HIP_R',
  'R_HIP_P',
  'R_KNEE',
  'R_ANKLE_P',
  'R_ANKLE_R',
  'L_HIP_Y',
  'L_HIP_R',
  'L_HIP_P',
  'L_KNEE',
  'L_ANKLE_P',
  'L_ANKLE_R'],
 'kp': 2000.0,
 'kd': 50.0,
 'termination_if_roll_greater_than': 150,
 'termination_if_pitch_greater_than': 150,
 'base_init_pos': [0.0, 0.0, 0.64],
 'base_init_quat': [1.0, 0.0, 0.0, 0.0],
 'episode_length_s': 20.0,
 'resampling_time_s': 4.0,
 'action_scale': 1.0,
 'simulate_action_latency': True,
 'clip_actions': 100.0,
 'dt': 0.01,
 'substeps': 10,
 'rotorInertia': 0.1,
 'base_roll_noise': [0, 0],
 'base_pitch_noise': [0, 0],
 'domain_rand': {'friction': [0.4, 1.1],
  'restitution': [0.0, 0.2],
  'kp': [1800.0, 2200.0],
  'kd': [25.0, 100.0]}

In [11]:
env = RLEnv(
    num_envs=1,
    env_cfg=env_cfg,
    obs_cfg=obs_cfg,
    reward_cfg=reward_cfg,
    command_cfg=command_cfg,
    dt=env_cfg['dt'],
    substeps=env_cfg['substeps'],
    show_viewer=True,
    robot_urdf_path=robot_path,
)

In [12]:
runner = OnPolicyRunner(env, train_cfg, log_dir, device='cuda')
resume_path = os.path.join(log_dir, f"model_{ckpt}.pt")
runner.load(resume_path)
policy = runner.get_inference_policy(device='cuda')

obs, _ = env.reset()
cnt = 0

torques = env.sim.sbody.getTorques()

print("obs : ", obs["policy"])

# データを記録
step_data.append(cnt)
obs_data.append(_obs_vec(obs))
torque_data.append(torques.copy())

cnt += 1

--------------------------------------------------------------------------------
Resolved observation sets: 
	 policy :  ['policy']
	 critic :  ['policy']
--------------------------------------------------------------------------------
Actor MLP: MLP(
  (0): Linear(in_features=45, out_features=512, bias=True)
  (1): ELU(alpha=1.0)
  (2): Linear(in_features=512, out_features=256, bias=True)
  (3): ELU(alpha=1.0)
  (4): Linear(in_features=256, out_features=128, bias=True)
  (5): ELU(alpha=1.0)
  (6): Linear(in_features=128, out_features=12, bias=True)
)
Critic MLP: MLP(
  (0): Linear(in_features=45, out_features=512, bias=True)
  (1): ELU(alpha=1.0)
  (2): Linear(in_features=512, out_features=256, bias=True)
  (3): ELU(alpha=1.0)
  (4): Linear(in_features=256, out_features=128, bias=True)
  (5): ELU(alpha=1.0)
  (6): Linear(in_features=128, out_features=1, bias=True)
)
obs :  tensor([[0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0.,
        

In [13]:
with torch.no_grad():
    print("cnt :", cnt)   
    actions = policy(obs)

    # アクションに倍率を適用して動きを制限
    scaled_actions = actions * action_scale

    print("Original actions : ", actions)
    print("Scaled actions : ", scaled_actions)

    obs, rews, dones, infos = env.step(scaled_actions) # スケール済みアクションを使用
    print("obs : ", obs["policy"])
    torques = env.sim.sbody.getTorques()
    print("torques:", torques)
    
    # データを記録
    step_data.append(cnt)
    # action_data.append(actions.cpu().numpy().flatten())
    obs_data.append(_obs_vec(obs))
    torque_data.append(torques.copy())
    
    cnt += 1

print(f"データ収集: step {cnt}")
     


cnt : 1
Original actions :  tensor([[-0.1771, -0.0822, -0.3819,  0.1484, -1.1932,  0.1420,  0.0718,  0.1889,
          0.5174,  0.1196, -1.0868,  0.4054]], device='cuda:0')
Scaled actions :  tensor([[-0.1771, -0.0822, -0.3819,  0.1484, -1.1932,  0.1420,  0.0718,  0.1889,
          0.5174,  0.1196, -1.0868,  0.4054]], device='cuda:0')


/userdir/irsl_rl/rl_env_base.py:110: UserWarning: To copy construct from a tensor, it is recommended to use sourceTensor.clone().detach() or sourceTensor.clone().detach().requires_grad_(True), rather than torch.tensor(sourceTensor).
  self.exact_actions = torch.tensor(actions, device=self.device, dtype=torch.float32) ## copy
/userdir/irsl_rl/rl_env_cnoid.py:103: UserWarning: Creating a tensor from a list of numpy.ndarrays is extremely slow. Please consider converting the list to a single numpy.ndarray with numpy.array() before converting to a tensor. (Triggered internally at /pytorch/torch/csrc/utils/tensor_new.cpp:254.)
  self.dof_pos = torch.tensor([self.convAnglesToGenesis(sbody.angleVector())]).to(torch.float32).to(self.device)


obs :  tensor([[-4.2367e-06, -7.9481e-03, -1.6194e-06,  5.0711e-10,  2.0808e-20,
         -1.0000e+00,  1.0000e+00,  0.0000e+00,  0.0000e+00,  1.4362e-08,
          3.0845e-08, -1.4168e-04,  3.6454e-04, -1.9103e-04, -3.4389e-08,
         -8.5621e-08, -4.7636e-08, -1.4180e-04,  3.6466e-04, -1.9103e-04,
          7.6633e-07,  7.1809e-07,  1.5422e-06, -7.0859e-03,  1.8227e-02,
         -9.5517e-03, -1.7194e-06, -4.2811e-06, -2.3818e-06, -7.0895e-03,
          1.8232e-02, -9.5524e-03,  3.8316e-05, -1.7705e-01, -8.2219e-02,
         -3.8193e-01,  1.4836e-01, -1.1932e+00,  1.4199e-01,  7.1801e-02,
          1.8885e-01,  5.1738e-01,  1.1963e-01, -1.0868e+00,  4.0544e-01]],
       device='cuda:0')
torques: [ 3.99683909e-16 -9.12643102e-16  2.42782550e-06  7.51007967e-06
  1.66224901e-06  2.18425700e-16  2.17253195e-16 -6.61016595e-16
  2.42782550e-06  7.51007967e-06  1.66224901e-06 -7.93165041e-17]
データ収集: step 2


In [14]:
with torch.no_grad():
    print("cnt :", cnt)   
    actions = policy(obs)

    # アクションに倍率を適用して動きを制限
    scaled_actions = actions * action_scale

    print("Original actions : ", actions)
    print("Scaled actions : ", scaled_actions)

    obs, rews, dones, infos = env.step(scaled_actions) # スケール済みアクションを使用
    print("obs : ", obs["policy"])
    torques = env.sim.sbody.getTorques()
    print("torques:", torques)
    
    # データを記録
    step_data.append(cnt)
    # action_data.append(actions.cpu().numpy().flatten())
    obs_data.append(_obs_vec(obs))
    torque_data.append(torques.copy())
    
    cnt += 1

print(f"データ収集: step {cnt}")
     


cnt : 2
Original actions :  tensor([[-0.1805, -0.2767, -0.4605,  0.5014, -1.4378, -0.1637, -0.1219,  0.4007,
          0.1474,  0.4558, -1.5629,  0.6768]], device='cuda:0')
Scaled actions :  tensor([[-0.1805, -0.2767, -0.4605,  0.5014, -1.4378, -0.1637, -0.1219,  0.4007,
          0.1474,  0.4558, -1.5629,  0.6768]], device='cuda:0')
obs :  tensor([[-1.1539e-01, -9.2509e-02,  3.7398e-01, -2.4949e-03,  2.6280e-03,
         -9.9999e-01,  1.0000e+00,  0.0000e+00,  0.0000e+00, -4.3693e-02,
         -1.0322e-03, -2.0019e-02,  3.9934e-02, -1.0768e-01,  9.5210e-02,
          2.9498e-02,  2.7146e-03,  1.6930e-03,  3.0208e-02, -1.0349e-01,
          7.0765e-02, -2.9668e-01, -1.2249e-02, -1.5259e-01,  2.8431e-01,
         -9.6812e-01,  5.8790e-01,  1.6108e-01,  3.8087e-02,  4.2018e-02,
          2.0607e-01, -9.2727e-01,  4.6146e-01, -1.8051e-01, -2.7669e-01,
         -4.6055e-01,  5.0135e-01, -1.4378e+00, -1.6368e-01, -1.2192e-01,
          4.0068e-01,  1.4736e-01,  4.5581e-01, -1.5629e+00,  6.7

In [15]:
with torch.no_grad():
    print("cnt :", cnt)   
    actions = policy(obs)

    # アクションに倍率を適用して動きを制限
    scaled_actions = actions * action_scale

    print("Original actions : ", actions)
    print("Scaled actions : ", scaled_actions)

    obs, rews, dones, infos = env.step(scaled_actions) # スケール済みアクションを使用
    print("obs : ", obs["policy"])
    torques = env.sim.sbody.getTorques()
    print("torques:", torques)
    
    # データを記録
    step_data.append(cnt)
    # action_data.append(actions.cpu().numpy().flatten())
    obs_data.append(_obs_vec(obs))
    torque_data.append(torques.copy())
    
    cnt += 1

print(f"データ収集: step {cnt}")
     


cnt : 3
Original actions :  tensor([[ 0.6248, -0.4005,  0.4347, -0.5803, -0.3123, -0.9677,  0.0711,  0.0354,
         -0.1294, -0.6332, -0.3795,  1.0005]], device='cuda:0')
Scaled actions :  tensor([[ 0.6248, -0.4005,  0.4347, -0.5803, -0.3123, -0.9677,  0.0711,  0.0354,
         -0.1294, -0.6332, -0.3795,  1.0005]], device='cuda:0')
obs :  tensor([[-0.1357, -0.1706,  0.7581, -0.0080,  0.0077, -0.9999,  1.0000,  0.0000,
          0.0000, -0.0965, -0.0103, -0.0692,  0.1288, -0.3873,  0.1078,  0.0079,
          0.0190,  0.0159,  0.0899, -0.3838,  0.2274, -0.1986, -0.0766, -0.3199,
          0.5670, -1.7305, -0.3657, -0.2533,  0.1180,  0.0961,  0.3589, -1.7831,
          0.9470,  0.6248, -0.4005,  0.4347, -0.5803, -0.3123, -0.9677,  0.0711,
          0.0354, -0.1294, -0.6332, -0.3795,  1.0005]], device='cuda:0')
torques: [  31.51834635 -200.         -200.          200.         -200.
 -200.          -29.1027983   200.          175.86692306  200.
 -200.          -25.36533361]
データ収集: step 4


In [16]:
with torch.no_grad():
    print("cnt :", cnt)   
    actions = policy(obs)

    # アクションに倍率を適用して動きを制限
    scaled_actions = actions * action_scale

    print("Original actions : ", actions)
    print("Scaled actions : ", scaled_actions)

    obs, rews, dones, infos = env.step(scaled_actions) # スケール済みアクションを使用
    print("obs : ", obs["policy"])
    torques = env.sim.sbody.getTorques()
    print("torques:", torques)
    
    # データを記録
    step_data.append(cnt)
    # action_data.append(actions.cpu().numpy().flatten())
    obs_data.append(_obs_vec(obs))
    torque_data.append(torques.copy())
    
    cnt += 1

print(f"データ収集: step {cnt}")
     


cnt : 4
Original actions :  tensor([[-0.0597,  0.7070,  0.3229, -0.5717,  0.5621, -0.0779,  0.7574, -0.4651,
         -0.6895, -1.1239,  0.6845, -0.1036]], device='cuda:0')
Scaled actions :  tensor([[-0.0597,  0.7070,  0.3229, -0.5717,  0.5621, -0.0779,  0.7574, -0.4651,
         -0.6895, -1.1239,  0.6845, -0.1036]], device='cuda:0')
obs :  tensor([[ 0.1909, -0.1224, -0.0168, -0.0137,  0.0057, -0.9999,  1.0000,  0.0000,
          0.0000, -0.0721, -0.0491, -0.1095,  0.2037, -0.6286, -0.0689,  0.0099,
          0.0280,  0.0286,  0.1438, -0.6322,  0.4738,  0.3800, -0.2889, -0.1003,
          0.2223, -0.7784, -1.3082,  0.1246,  0.0056,  0.0355,  0.2036, -0.8002,
          1.1511, -0.0597,  0.7070,  0.3229, -0.5717,  0.5621, -0.0779,  0.7574,
         -0.4651, -0.6895, -1.1239,  0.6845, -0.1036]], device='cuda:0')
torques: [ 200.         -200.          200.         -200.          200.
 -200.           10.08044394    2.83773009 -200.         -200.
  200.          -99.17608737]
データ収集: step 5


In [17]:
with torch.no_grad():
    print("cnt :", cnt)   
    actions = policy(obs)

    # アクションに倍率を適用して動きを制限
    scaled_actions = actions * action_scale

    print("Original actions : ", actions)
    print("Scaled actions : ", scaled_actions)

    obs, rews, dones, infos = env.step(scaled_actions) # スケール済みアクションを使用
    print("obs : ", obs["policy"])
    torques = env.sim.sbody.getTorques()
    print("torques:", torques)
    
    # データを記録
    step_data.append(cnt)
    # action_data.append(actions.cpu().numpy().flatten())
    obs_data.append(_obs_vec(obs))
    torque_data.append(torques.copy())
    
    cnt += 1

print(f"データ収集: step {cnt}")
     


cnt : 5
Original actions :  tensor([[-0.9501,  0.7894,  0.2835, -0.5856,  0.1804,  1.0416, -0.3645, -0.0124,
          0.7431, -0.6552,  0.5837, -0.8250]], device='cuda:0')
Scaled actions :  tensor([[-0.9501,  0.7894,  0.2835, -0.5856,  0.1804,  1.0416, -0.3645, -0.0124,
          0.7431, -0.6552,  0.5837, -0.8250]], device='cuda:0')
obs :  tensor([[ 0.2568, -0.0330, -0.3120, -0.0166, -0.0035, -0.9999,  1.0000,  0.0000,
          0.0000, -0.0372, -0.0979, -0.1113,  0.2138, -0.6801, -0.2273,  0.0908,
          0.0212,  0.0281,  0.1695, -0.6847,  0.6014,  0.0359, -0.2076,  0.0597,
         -0.0867,  0.1677, -0.3695,  0.6344, -0.0670, -0.0323,  0.0706,  0.1778,
          0.2182, -0.9501,  0.7894,  0.2835, -0.5856,  0.1804,  1.0416, -0.3645,
         -0.0124,  0.7431, -0.6552,  0.5837, -0.8250]], device='cuda:0')
torques: [ -98.30389529  200.          200.         -200.          200.
  200.          200.         -200.         -200.         -200.
  200.         -200.        ]
データ収集: step 6


In [18]:
with torch.no_grad():
    print("cnt :", cnt)   
    actions = policy(obs)

    # アクションに倍率を適用して動きを制限
    scaled_actions = actions * action_scale

    print("Original actions : ", actions)
    print("Scaled actions : ", scaled_actions)

    obs, rews, dones, infos = env.step(scaled_actions) # スケール済みアクションを使用
    print("obs : ", obs["policy"])
    torques = env.sim.sbody.getTorques()
    print("torques:", torques)
    
    # データを記録
    step_data.append(cnt)
    # action_data.append(actions.cpu().numpy().flatten())
    obs_data.append(_obs_vec(obs))
    torque_data.append(torques.copy())
    
    cnt += 1

print(f"データ収集: step {cnt}")
     


cnt : 6
Original actions :  tensor([[ 0.6623, -0.1507, -0.9873,  0.3802, -0.5670, -0.1835, -1.0428,  0.6101,
         -0.3289, -0.7735, -0.2790, -0.2103]], device='cuda:0')
Scaled actions :  tensor([[ 0.6623, -0.1507, -0.9873,  0.3802, -0.5670, -0.1835, -1.0428,  0.6101,
         -0.3289, -0.7735, -0.2790, -0.2103]], device='cuda:0')
obs :  tensor([[ 0.1218, -0.6098,  0.4002, -0.0307, -0.0103, -0.9995,  1.0000,  0.0000,
          0.0000, -0.0884, -0.1242, -0.0691,  0.1648, -0.5425, -0.1980,  0.1551,
          0.0177,  0.0653,  0.1506, -0.5438,  0.5385, -0.4944, -0.0816,  0.3313,
         -0.3788,  1.1140,  0.5682,  0.0647,  0.0161,  0.3609, -0.2329,  1.1355,
         -0.7501,  0.6623, -0.1507, -0.9873,  0.3802, -0.5670, -0.1835, -1.0428,
          0.6101, -0.3289, -0.7735, -0.2790, -0.2103]], device='cuda:0')
torques: [-200.          200.          200.         -200.          200.
  200.         -200.          -70.05227993  200.         -200.
  200.         -200.        ]
データ収集: step 7


In [19]:
with torch.no_grad():
    print("cnt :", cnt)   
    actions = policy(obs)

    # アクションに倍率を適用して動きを制限
    scaled_actions = actions * action_scale

    print("Original actions : ", actions)
    print("Scaled actions : ", scaled_actions)

    obs, rews, dones, infos = env.step(scaled_actions) # スケール済みアクションを使用
    print("obs : ", obs["policy"])
    torques = env.sim.sbody.getTorques()
    print("torques:", torques)
    
    # データを記録
    step_data.append(cnt)
    # action_data.append(actions.cpu().numpy().flatten())
    obs_data.append(_obs_vec(obs))
    torque_data.append(torques.copy())
    
    cnt += 1

print(f"データ収集: step {cnt}")
     


cnt : 7
Original actions :  tensor([[ 0.8906,  0.2142, -0.7886,  0.9028, -0.4810, -0.0656,  0.1065, -0.0228,
         -0.3123,  0.1749, -0.3278, -0.0996]], device='cuda:0')
Scaled actions :  tensor([[ 0.8906,  0.2142, -0.7886,  0.9028, -0.4810, -0.0656,  0.1065, -0.0228,
         -0.3123,  0.1749, -0.3278, -0.0996]], device='cuda:0')
obs :  tensor([[-0.3738, -0.1669,  0.4139, -0.0463, -0.0032, -0.9989,  1.0000,  0.0000,
          0.0000, -0.1388, -0.1301, -0.0487,  0.1469, -0.4236, -0.1382,  0.1130,
          0.0489,  0.0980,  0.1276, -0.3561,  0.3215, -0.0539,  0.0165, -0.0785,
          0.1316,  0.1241, -0.0439, -0.4066,  0.2344,  0.0603, -0.1099,  0.6146,
         -1.2023,  0.8906,  0.2142, -0.7886,  0.9028, -0.4810, -0.0656,  0.1065,
         -0.0228, -0.3123,  0.1749, -0.3278, -0.0996]], device='cuda:0')
torques: [ 200.          -32.33599951 -200.          200.         -200.
 -200.         -200.          200.         -200.         -200.
 -200.          200.        ]
データ収集: step 8


In [20]:
with torch.no_grad():
    print("cnt :", cnt)   
    actions = policy(obs)

    # アクションに倍率を適用して動きを制限
    scaled_actions = actions * action_scale

    print("Original actions : ", actions)
    print("Scaled actions : ", scaled_actions)

    obs, rews, dones, infos = env.step(scaled_actions) # スケール済みアクションを使用
    print("obs : ", obs["policy"])
    torques = env.sim.sbody.getTorques()
    print("torques:", torques)
    
    # データを記録
    step_data.append(cnt)
    # action_data.append(actions.cpu().numpy().flatten())
    obs_data.append(_obs_vec(obs))
    torque_data.append(torques.copy())
    
    cnt += 1

print(f"データ収集: step {cnt}")
     


cnt : 8
Original actions :  tensor([[-1.0072, -0.6858,  0.2638, -0.5152, -0.6204,  0.5146,  0.3574, -1.8984,
          0.5103, -1.5368, -1.0913,  0.2879]], device='cuda:0')
Scaled actions :  tensor([[-1.0072, -0.6858,  0.2638, -0.5152, -0.6204,  0.5146,  0.3574, -1.8984,
          0.5103, -1.5368, -1.0913,  0.2879]], device='cuda:0')
obs :  tensor([[-0.4797,  0.5202, -0.2217, -0.0377,  0.0143, -0.9992,  1.0000,  0.0000,
          0.0000, -0.0989, -0.1172, -0.1007,  0.2060, -0.4371, -0.1169,  0.0722,
          0.0913,  0.0692,  0.1283, -0.2898,  0.1164,  0.4039,  0.0999, -0.4112,
          0.4234, -0.0917,  0.1106,  0.0140,  0.1890, -0.3074,  0.0779, -0.0288,
         -0.5928, -1.0072, -0.6858,  0.2638, -0.5152, -0.6204,  0.5146,  0.3574,
         -1.8984,  0.5103, -1.5368, -1.0913,  0.2879]], device='cuda:0')
torques: [ 200.          200.         -200.          200.            3.7821132
   -8.04163138   86.65906087 -200.         -200.           29.79439521
  -91.24399337  200.        ]

In [21]:
with torch.no_grad():
    print("cnt :", cnt)   
    actions = policy(obs)

    # アクションに倍率を適用して動きを制限
    scaled_actions = actions * action_scale

    print("Original actions : ", actions)
    print("Scaled actions : ", scaled_actions)

    obs, rews, dones, infos = env.step(scaled_actions) # スケール済みアクションを使用
    print("obs : ", obs["policy"])
    torques = env.sim.sbody.getTorques()
    print("torques:", torques)
    
    # データを記録
    step_data.append(cnt)
    # action_data.append(actions.cpu().numpy().flatten())
    obs_data.append(_obs_vec(obs))
    torque_data.append(torques.copy())
    
    cnt += 1

print(f"データ収集: step {cnt}")
     


cnt : 9
Original actions :  tensor([[-0.2152,  0.4887,  0.8800, -0.1006,  0.0116, -0.1466, -0.3270,  0.1456,
          1.0303, -1.0678,  0.4531,  0.1896]], device='cuda:0')
Scaled actions :  tensor([[-0.2152,  0.4887,  0.8800, -0.1006,  0.0116, -0.1466, -0.3270,  0.1456,
          1.0303, -1.0678,  0.4531,  0.1896]], device='cuda:0')
obs :  tensor([[ 1.5554e-01, -1.3131e-01, -1.9169e-01, -3.1310e-02,  1.9488e-02,
         -9.9932e-01,  1.0000e+00,  0.0000e+00,  0.0000e+00, -6.2531e-02,
         -1.2754e-01, -1.4386e-01,  2.5222e-01, -5.0246e-01, -4.9955e-04,
          1.1159e-01,  1.0317e-01,  4.7075e-02,  1.4298e-01, -3.7832e-01,
          9.6023e-02,  2.7046e-03, -1.7224e-01, -5.1308e-02,  7.5843e-02,
         -4.0014e-01,  9.2261e-01,  3.1625e-01, -4.0039e-02,  4.9572e-02,
          1.3371e-01, -7.3285e-01,  2.7529e-01, -2.1524e-01,  4.8867e-01,
          8.7996e-01, -1.0064e-01,  1.1572e-02, -1.4664e-01, -3.2698e-01,
          1.4561e-01,  1.0303e+00, -1.0678e+00,  4.5306e-01,  1.8

In [22]:
with torch.no_grad():
    print("cnt :", cnt)   
    actions = policy(obs)

    # アクションに倍率を適用して動きを制限
    scaled_actions = actions * action_scale

    print("Original actions : ", actions)
    print("Scaled actions : ", scaled_actions)

    obs, rews, dones, infos = env.step(scaled_actions) # スケール済みアクションを使用
    print("obs : ", obs["policy"])
    torques = env.sim.sbody.getTorques()
    print("torques:", torques)
    
    # データを記録
    step_data.append(cnt)
    # action_data.append(actions.cpu().numpy().flatten())
    obs_data.append(_obs_vec(obs))
    torque_data.append(torques.copy())
    
    cnt += 1

print(f"データ収集: step {cnt}")
     


cnt : 10
Original actions :  tensor([[ 0.3305, -0.0802, -1.3266,  0.0466, -0.3191, -0.6531, -0.5003, -0.2192,
         -1.5590, -0.7820,  0.0080, -0.3244]], device='cuda:0')
Scaled actions :  tensor([[ 0.3305, -0.0802, -1.3266,  0.0466, -0.3191, -0.6531, -0.5003, -0.2192,
         -1.5590, -0.7820,  0.0080, -0.3244]], device='cuda:0')
obs :  tensor([[-0.1490, -0.7567,  0.4245, -0.0501,  0.0206, -0.9985,  1.0000,  0.0000,
          0.0000, -0.1093, -0.1375, -0.1218,  0.2379, -0.4779,  0.0735,  0.1134,
          0.1104,  0.0977,  0.1443, -0.4219,  0.1254, -0.3379,  0.0326,  0.2550,
         -0.1917,  0.5508, -0.0797, -0.2308,  0.0766,  0.4072, -0.0840,  0.1943,
          0.1105,  0.3305, -0.0802, -1.3266,  0.0466, -0.3191, -0.6531, -0.5003,
         -0.2192, -1.5590, -0.7820,  0.0080, -0.3244]], device='cuda:0')
torques: [ 200.          200.          200.         -200.          200.
 -200.         -200.            3.3679241   200.         -200.
  200.           19.78953763]
データ収集: step 1

In [23]:
with torch.no_grad():
    print("cnt :", cnt)   
    actions = policy(obs)

    # アクションに倍率を適用して動きを制限
    scaled_actions = actions * action_scale

    print("Original actions : ", actions)
    print("Scaled actions : ", scaled_actions)

    obs, rews, dones, infos = env.step(scaled_actions) # スケール済みアクションを使用
    print("obs : ", obs["policy"])
    torques = env.sim.sbody.getTorques()
    print("torques:", torques)
    
    # データを記録
    step_data.append(cnt)
    # action_data.append(actions.cpu().numpy().flatten())
    obs_data.append(_obs_vec(obs))
    torque_data.append(torques.copy())
    
    cnt += 1

print(f"データ収集: step {cnt}")
     


cnt : 11
Original actions :  tensor([[ 0.5478, -0.0238, -0.2493,  0.6348, -0.7120,  0.5158,  0.5657, -0.4513,
         -0.4309, -0.6165, -0.1358, -0.7650]], device='cuda:0')
Scaled actions :  tensor([[ 0.5478, -0.0238, -0.2493,  0.6348, -0.7120,  0.5158,  0.5657, -0.4513,
         -0.4309, -0.6165, -0.1358, -0.7650]], device='cuda:0')
obs :  tensor([[-0.0489, -0.0393,  0.3661, -0.0640,  0.0253, -0.9976,  1.0000,  0.0000,
          0.0000, -0.1358, -0.1308, -0.0925,  0.1910, -0.4083, -0.0493,  0.0153,
          0.1077,  0.1583,  0.1142, -0.3037,  0.0433,  0.0393,  0.0348,  0.0496,
         -0.2410,  0.2018, -1.0498, -0.7039, -0.0904,  0.2209, -0.2002,  0.6784,
         -0.7480,  0.5478, -0.0238, -0.2493,  0.6348, -0.7120,  0.5158,  0.5657,
         -0.4513, -0.4309, -0.6165, -0.1358, -0.7650]], device='cuda:0')
torques: [ 200.           67.40658343 -200.          -59.93964576  -26.26323347
 -200.         -200.         -200.         -200.         -200.
  -50.56037258  -34.58051826]
データ収集

In [64]:
# 既存のforループを置き換え
num_steps = 10
for i in range(num_steps):
    with torch.no_grad():
        actions = policy(obs)
        
        # アクションスケーリング
        scaled_actions = actions * action_scale
        obs, rews, dones, infos = env.step(scaled_actions)  # スケール済みを使用
        torques = env.sim.sbody.getTorques()
        
        # データを記録
        step_data.append(cnt)
        obs_data.append(_obs_vec(obs))
        torque_data.append(torques.copy())
        
        # デバッグ表示（最初の数ステップのみ）
        if i < 3:
            print(f"Step {i}: Original action max={actions.max():.3f}, "
                  f"Scaled action max={scaled_actions.max():.3f}")
        
        if i % 20 == 0:
            print(f"Step {i+1}/{num_steps}, Total steps: {cnt}")
            print("steps:",cnt)
            print("actions :",scaled_actions)
            print("target_dof_pos:",env.target_dof_pos)
        
        cnt += 1

print(f"データ収集完了: {num_steps} steps collected with action_scale={action_scale}")

Step 0: Original action max=0.765, Scaled action max=0.765
Step 1/10, Total steps: 392
steps: 392
actions : tensor([[-0.2428, -1.2518, -0.6750, -0.0622, -0.2570,  0.0704, -0.1178,  0.4516,
         -0.5102, -0.3979, -0.5371,  0.7646]], device='cuda:0')
target_dof_pos: tensor([[-0.2059, -1.2263, -0.6363,  2.1013, -1.6491, -0.1130, -0.1864,  1.2120,
         -0.9237,  1.6004, -1.7201,  1.6002]], device='cuda:0')
Step 1: Original action max=0.300, Scaled action max=0.300
Step 2: Original action max=1.080, Scaled action max=1.080
データ収集完了: 10 steps collected with action_scale=1.0


In [65]:
# for i in range(500):
#     with torch.no_grad():
#         actions = policy(obs)
#         scaled_actions = actions * action_scale
#         obs, rews, dones, infos = env.step(scaled_actions)

In [66]:
env.sim.stop()

In [44]:
# 最もシンプルな保存方法
def save_simple_csv():
    if not step_data:
        print("データがありません")
        return
    
    # 基本的な辞書形式でデータを整理
    data_dict = {'step': step_data}
    
    # # Actionデータ
    # action_array = np.array(action_data)
    # for i in range(action_array.shape[1]):
    #     data_dict[f'action_{i}'] = action_array[:, i]
    
    # Observationデータ
    obs_array = np.array(obs_data)
    for i in range(obs_array.shape[1]):
        data_dict[f'obs_{i}'] = obs_array[:, i]
    
    # Torqueデータ
    torque_array = np.array(torque_data)
    for i in range(torque_array.shape[1]):
        data_dict[f'torque_{i}'] = torque_array[:, i]
    
    # DataFrameを作成して保存
    df = pd.DataFrame(data_dict)
    csv_filename = f'obs_data/cnoid_{exp_name}_ckpt{ckpt}_scale{action_scale}_rotorInertia0.9.csv'
    df.to_csv(csv_filename, index=False)
    
    print(f"シンプル版を保存: {csv_filename}")
    print(f"データ形状: {df.shape}")
    
    return df

# シンプル版を実行
df_simple = save_simple_csv()

シンプル版を保存: obs_data/cnoid_friction-walking-fractal-kp2000kd50_ckpt100_scale1.0_rotorInertia0.9.csv
データ形状: (192, 58)
